<a href="https://colab.research.google.com/github/JustinCho-Soungbin/ad-performance-insight-engine/blob/main/Ads_Cross_Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install & Imports

In [ ]:
!pip install xgboost shap optuna -q

In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (classification_report, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import shap
import optuna
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully!")

# Load Data

In [ ]:
# =============================================================
# DATA LOADING FROM KAGGLE
# Using kagglehub to directly download the Global Ads Performance
# dataset containing cross-platform advertising metrics from
# Google Ads, Meta Ads, and TikTok Ads.
# =============================================================

!pip install kagglehub -q

import kagglehub
import pandas as pd
import os

# Download dataset
path = kagglehub.dataset_download("nudratabbas/global-ads-performance-google-meta-tiktok")
print("Path to dataset files:", path)

# Check what files are in the folder
print(os.listdir(path))

In [ ]:
# Load dataset from downloaded path
df = pd.read_csv(f"{path}/global_ads_performance_dataset.csv")

print(f"Dataset shape: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")
df.head(3)

# EDA


In [ ]:
# =============================================================
# EXPLORATORY DATA ANALYSIS (EDA)
# Before building any model, we explore the data to understand:
# - Distribution of key metrics (ROAS, CTR, CPC)
# - Performance differences across platforms
# This step ensures we make informed decisions during
# feature engineering and model design.
# =============================================================

import matplotlib.pyplot as plt
import seaborn as sns

# --- Basic stats ---
print("=== ROAS by Platform ===")
print(df.groupby('platform')['ROAS'].describe().round(2))

print("\n=== ROAS by Campaign Type ===")
print(df.groupby('campaign_type')['ROAS'].mean().sort_values(ascending=False).round(2))

print("\n=== ROAS by Industry ===")
print(df.groupby('industry')['ROAS'].mean().sort_values(ascending=False).round(2))

# --- Visualize ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. ROAS distribution
axes[0,0].hist(df['ROAS'], bins=50, color='steelblue', edgecolor='white')
axes[0,0].axvline(df['ROAS'].median(), color='red', linestyle='--',
                   label=f'Median: {df["ROAS"].median():.2f}')
axes[0,0].set_title('ROAS Distribution')
axes[0,0].set_xlabel('ROAS')
axes[0,0].legend()

# 2. ROAS by platform
df.groupby('platform')['ROAS'].mean().sort_values().plot(
    kind='barh', ax=axes[0,1], color='steelblue')
axes[0,1].set_title('Avg ROAS by Platform')
axes[0,1].set_xlabel('ROAS')

# 3. ROAS by campaign type
df.groupby('campaign_type')['ROAS'].mean().sort_values().plot(
    kind='barh', ax=axes[1,0], color='steelblue')
axes[1,0].set_title('Avg ROAS by Campaign Type')
axes[1,0].set_xlabel('ROAS')

# 4. ROAS by industry
df.groupby('industry')['ROAS'].mean().sort_values().plot(
    kind='barh', ax=axes[1,1], color='steelblue')
axes[1,1].set_title('Avg ROAS by Industry')
axes[1,1].set_xlabel('ROAS')

plt.suptitle('Global Ads Performance — EDA', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Feature Engineering


In [ ]:
# =============================================================
# FEATURE ENGINEERING
# Raw ad metrics alone are not sufficient for modeling.
# We derive additional behavioral and contextual signals
# that better capture the quality and efficiency of ad spend.
#
# Derived features:
# - conversion_rate : conversions / clicks
#   → how effectively clicks turn into actual conversions
# - spend_efficiency: revenue / ad_spend
#   → direct measure of ad profitability (similar to ROAS)
# - ctr_cpc_ratio   : CTR / CPC
#   → captures cost-effectiveness of audience targeting
# - log_impressions : log(impressions)
#   → reduces skew from large impression counts
# - cost_per_impression: ad_spend / impressions
#   → how much we pay per eyeball
#
# Categorical features (platform, campaign_type, industry, country)
# are label-encoded for XGBoost compatibility.
#
# Target variable:
# high_roas = 1 if ROAS > median (4.295), else 0
# → "Is this ad campaign worth scaling?"
# =============================================================

import numpy as np
from sklearn.preprocessing import LabelEncoder

df_model = df.copy()

# --- Derived features ---
df_model['conversion_rate']     = df_model['conversions'] / df_model['clicks']
df_model['spend_efficiency']    = df_model['revenue'] / df_model['ad_spend']
df_model['ctr_cpc_ratio']       = df_model['CTR'] / (df_model['CPC'] + 1e-6)
df_model['log_impressions']     = np.log1p(df_model['impressions'])
df_model['cost_per_impression'] = df_model['ad_spend'] / df_model['impressions']

# --- Target variable ---
ROAS_THRESHOLD = df['ROAS'].median()
df_model['high_roas'] = (df_model['ROAS'] > ROAS_THRESHOLD).astype(int)

print(f"ROAS threshold (median): {ROAS_THRESHOLD:.3f}")
print(f"Class distribution:\n{df_model['high_roas'].value_counts()}")

# --- Encode categoricals ---
cat_cols = ['platform', 'campaign_type', 'industry', 'country']
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df_model[col + '_enc'] = le.fit_transform(df_model[col])
    le_dict[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# --- Final feature set ---
# Note: ROAS itself is excluded to prevent data leakage
# spend_efficiency is also excluded (revenue/ad_spend ≈ ROAS)
FEATURES = [
    # Original ad metrics
    'impressions', 'clicks', 'CTR', 'CPC', 'ad_spend', 'conversions', 'CPA',
    # Derived behavioral signals
    'conversion_rate', 'ctr_cpc_ratio',
    'log_impressions', 'cost_per_impression',
    # Encoded contextual features
    'platform_enc', 'campaign_type_enc', 'industry_enc', 'country_enc'
]

TARGET = 'high_roas'

print(f"\nTotal features: {len(FEATURES)}")
print(f"Features used: {FEATURES}")

# Model Selection + Target Analysis

In [ ]:
# =============================================================
# MODEL SELECTION
# Before jumping into XGBoost, we benchmark multiple baseline
# models to understand which algorithm best fits our data.
# This is standard ML practice — never assume one model wins.
#
# Models we compare:
# - Logistic Regression : linear baseline, interpretable
# - Decision Tree       : non-linear, captures splits
# - Random Forest       : ensemble of trees, robust
# - XGBoost             : gradient boosting, usually strongest
# - KNN                 : distance-based, no assumptions
#
# Why compare?
# - Our data is tabular with mixed numeric + categorical features
# - Class is perfectly balanced (900/900) so accuracy is reliable
# - We want to justify XGBoost choice with evidence, not assumption
#
# Target: high_roas (binary)
# - 1 = ROAS > 4.295 (high performing campaign)
# - 0 = ROAS <= 4.295 (underperforming campaign)
# This framing answers: "Is this ad campaign worth scaling?"
# =============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt

X = df_model[FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size : {X_train.shape[0]}")
print(f"Test size  : {X_test.shape[0]}")

# --- Define models ---
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'      : DecisionTreeClassifier(random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost'            : xgb.XGBClassifier(n_estimators=100, random_state=42,
                                              eval_metric='logloss', verbosity=0),
    'KNN'                : KNeighborsClassifier(n_neighbors=5)
}

# --- Cross-validation (5-fold stratified) ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print("\n=== 5-Fold Cross Validation Results ===")
print(f"{'Model':<25} {'Accuracy':>10} {'ROC-AUC':>10} {'Std':>8}")
print("-" * 55)

for name, model in models.items():
    acc_scores = cross_val_score(model, X_train, y_train,
                                  cv=cv, scoring='accuracy')
    auc_scores = cross_val_score(model, X_train, y_train,
                                  cv=cv, scoring='roc_auc')
    results[name] = {
        'accuracy': acc_scores.mean(),
        'auc'     : auc_scores.mean(),
        'std'     : acc_scores.std()
    }
    print(f"{name:<25} {acc_scores.mean():>10.4f} {auc_scores.mean():>10.4f} {acc_scores.std():>8.4f}")

# --- Visualize comparison ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names    = list(results.keys())
accs     = [results[m]['accuracy'] for m in names]
aucs     = [results[m]['auc'] for m in names]
colors   = ['#d9534f' if m == 'XGBoost' else 'steelblue' for m in names]

# Accuracy bar
bars = axes[0].barh(names, accs, color=colors)
axes[0].set_xlim(0.5, 1.0)
axes[0].set_title('CV Accuracy by Model')
axes[0].set_xlabel('Accuracy')
for bar, val in zip(bars, accs):
    axes[0].text(val + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=10)

# ROC-AUC bar
bars2 = axes[1].barh(names, aucs, color=colors)
axes[1].set_xlim(0.5, 1.0)
axes[1].set_title('CV ROC-AUC by Model')
axes[1].set_xlabel('ROC-AUC')
for bar, val in zip(bars2, aucs):
    axes[1].text(val + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=10)

plt.suptitle('Model Comparison — 5-Fold Cross Validation',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

best_model = max(results, key=lambda x: results[x]['auc'])
print(f"\nBest model by ROC-AUC: {best_model} ({results[best_model]['auc']:.4f})")

# Logistic Regression Anaylze

In [ ]:
# =============================================================
# LOGISTIC REGRESSION — DETAILED EVALUATION
# Logistic Regression outperformed all other models in cross-
# validation (Accuracy: 0.7743, ROC-AUC: 0.8662).
# This is likely due to our relatively small dataset (1,800 rows)
# where simpler models generalize better than complex ones.
#
# Here we evaluate it on the held-out test set and analyze:
# - Classification report (Precision, Recall, F1)
# - ROC-AUC curve
# - Feature coefficients (which features matter most?)
# =============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay,
                             RocCurveDisplay)
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

# Logistic Regression needs feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Train
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr  = lr_model.predict_proba(X_test_scaled)[:, 1]

print("=== Logistic Regression — Test Set Performance ===")
print(classification_report(y_test, y_pred_lr,
                             target_names=['Low ROAS', 'High ROAS']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_lr):.4f}")

# --- Plots ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred_lr)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Low ROAS', 'High ROAS'])
disp.plot(cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix')

# 2. ROC Curve
RocCurveDisplay.from_predictions(y_test, y_prob_lr, ax=axes[1],
                                  name='Logistic Regression')
axes[1].plot([0,1], [0,1], 'k--', label='Random (AUC=0.5)')
axes[1].set_title('ROC Curve')
axes[1].legend()

# 3. Feature coefficients
coef_df = pd.DataFrame({
    'feature'    : FEATURES,
    'coefficient': lr_model.coef_[0]
}).sort_values('coefficient', key=abs, ascending=True)

colors = ['#d9534f' if c > 0 else 'steelblue' for c in coef_df['coefficient']]
axes[2].barh(coef_df['feature'], coef_df['coefficient'], color=colors)
axes[2].axvline(0, color='black', linewidth=0.8)
axes[2].set_title('Feature Coefficients\n(red=positive, blue=negative)')
axes[2].set_xlabel('Coefficient')

plt.suptitle('Logistic Regression — Detailed Evaluation',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================
# DATA LEAKAGE CHECK
# CPA (Cost Per Acquisition) = ad_spend / conversions
# This is mathematically correlated with ROAS (our target).
# A model that relies heavily on CPA is essentially "cheating"
# — it's using a derived metric that directly encodes the answer.
# We remove CPA and retrain to get a fairer evaluation.
# =============================================================

FEATURES_NO_CPA = [f for f in FEATURES if f != 'CPA']

print(f"Features without CPA: {len(FEATURES_NO_CPA)}")

X_no_cpa = df_model[FEATURES_NO_CPA]

X_train_nc, X_test_nc, y_train_nc, y_test_nc = train_test_split(
    X_no_cpa, y, test_size=0.2, random_state=42, stratify=y
)

# Scale
scaler_nc = StandardScaler()
X_train_nc_scaled = scaler_nc.fit_transform(X_train_nc)
X_test_nc_scaled  = scaler_nc.transform(X_test_nc)

# Retrain
lr_no_cpa = LogisticRegression(max_iter=1000, random_state=42)
lr_no_cpa.fit(X_train_nc_scaled, y_train_nc)

y_pred_nc = lr_no_cpa.predict(X_test_nc_scaled)
y_prob_nc  = lr_no_cpa.predict_proba(X_test_nc_scaled)[:, 1]

print("\n=== With CPA ===")
print(f"Accuracy : 0.7800")
print(f"ROC-AUC  : 0.8777")

print("\n=== Without CPA ===")
print(classification_report(y_test_nc, y_pred_nc,
                             target_names=['Low ROAS', 'High ROAS']))
print(f"ROC-AUC  : {roc_auc_score(y_test_nc, y_prob_nc):.4f}")

In [ ]:
# =============================================================
# LOGISTIC REGRESSION — RESULTS SUMMARY
#
# Performance on held-out test set (360 samples):
#   Accuracy : 0.78
#   ROC-AUC  : 0.8777
#
# Key Findings:
# 1. Logistic Regression outperformed all other baseline models
#    in cross-validation, suggesting that linear decision boundaries
#    are sufficient for this dataset size (1,800 rows).
#
# 2. CPA (Cost Per Acquisition) had the largest coefficient,
#    indicating it is the strongest predictor of high ROAS.
#    This makes intuitive sense:
#    lower CPA → more conversions per dollar → higher ROAS.
#
# 3. Data leakage check passed:
#    Removing CPA caused only a 0.0003 drop in ROC-AUC (0.8777 → 0.8774),
#    confirming CPA is a genuine signal, not a shortcut.
#
# 4. Class balance is perfect (180/180 in test set),
#    so accuracy is a reliable metric here.
#
# Limitation:
# - Logistic Regression assumes linear relationships between
#   features and the log-odds of the target.
# - Complex interactions (e.g. platform × campaign_type) may
#   not be captured by a linear model.
#
# Next Step:
# - Tune XGBoost with Optuna to capture non-linear interactions
#   and attempt to surpass the 0.8777 ROC-AUC baseline.
# =============================================================

print("Logistic Regression — Final Results")
print("=" * 40)
print(f"Accuracy : 0.7800")
print(f"ROC-AUC  : 0.8777")
print(f"Precision (High ROAS): 0.75")
print(f"Recall    (High ROAS): 0.83")
print(f"F1-Score  (High ROAS): 0.79")
print()
print("Strongest predictor : CPA (Cost Per Acquisition)")
print("Leakage check       : Passed (ΔAUC = 0.0003)")
print("Next                : XGBoost tuning with Optuna")


# XGBoost + Optuna

In [ ]:
# =============================================================
# XGBOOST HYPERPARAMETER TUNING WITH OPTUNA
#
# Goal: Surpass Logistic Regression baseline (ROC-AUC: 0.8777)
#
# Why Optuna?
# - Bayesian optimization: smarter than GridSearch
#   (learns from previous trials to find better params faster)
# - More efficient than exhaustive GridSearchCV
#
# Parameters we tune:
# - n_estimators    : number of trees
# - max_depth       : how deep each tree grows
# - learning_rate   : step size (lower = more careful)
# - subsample       : fraction of data per tree (prevents overfit)
# - colsample_bytree: fraction of features per tree
# - min_child_weight: minimum samples in a leaf node
# - gamma           : minimum loss reduction to make a split
#
# Evaluation: 5-fold stratified CV on training set
# Metric: ROC-AUC (same as baseline comparison)
# =============================================================

!pip install optuna -q

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

X_train_xgb = X_train  # no scaling needed for XGBoost
X_test_xgb  = X_test

def objective(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 100, 500),
        'max_depth'        : trial.suggest_int('max_depth', 3, 8),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight' : trial.suggest_int('min_child_weight', 1, 10),
        'gamma'            : trial.suggest_float('gamma', 0, 5),
        'random_state'     : 42,
        'eval_metric'      : 'logloss',
        'verbosity'        : 0
    }

    model = xgb.XGBClassifier(**params)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_xgb, y_train,
                             cv=cv, scoring='roc_auc')
    return scores.mean()

# Run optimization (100 trials)
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"\nBest ROC-AUC (CV): {study.best_value:.4f}")
print(f"Best params:\n{study.best_params}")

In [ ]:
# =============================================================
# XGBOOST TUNED — FINAL EVALUATION ON TEST SET
#
# Optuna found best CV ROC-AUC of 0.8633 (vs LR: 0.8777)
# XGBoost did not surpass Logistic Regression, likely due to
# small dataset size (1,800 rows). With limited data, simpler
# models tend to generalize better.
#
# We evaluate on held-out test set for final comparison.
# =============================================================

best_params = study.best_params
best_params['random_state'] = 42
best_params['eval_metric']  = 'logloss'
best_params['verbosity']    = 0

xgb_tuned = xgb.XGBClassifier(**best_params)
xgb_tuned.fit(X_train, y_train)

y_pred_xgb = xgb_tuned.predict(X_test)
y_prob_xgb = xgb_tuned.predict_proba(X_test)[:, 1]

print("=== XGBoost Tuned — Test Set Performance ===")
print(classification_report(y_test, y_pred_xgb,
                             target_names=['Low ROAS', 'High ROAS']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_xgb):.4f}")

# --- Final comparison plot ---
fig, ax = plt.subplots(figsize=(8, 6))

RocCurveDisplay.from_predictions(
    y_test, y_prob_lr, ax=ax, name=f'Logistic Regression (AUC=0.8777)')
RocCurveDisplay.from_predictions(
    y_test, y_prob_xgb, ax=ax, name=f'XGBoost Tuned (AUC={roc_auc_score(y_test, y_prob_xgb):.4f})')
ax.plot([0,1], [0,1], 'k--', label='Random (AUC=0.5)')
ax.set_title('ROC Curve Comparison — LR vs XGBoost')
ax.legend()
plt.show()

# XGBoost Tuning Results Summary

In [ ]:
# =============================================================
# XGBOOST TUNING — RESULTS SUMMARY
#
# After 100 trials of Bayesian Optimization via Optuna,
# XGBoost (tuned) achieved a CV ROC-AUC of 0.8633,
# which is slightly lower than Logistic Regression (0.8777).
#
# Why did XGBoost underperform despite tuning?
#
# 1. Dataset size limitation:
#    With only 1,800 rows, there is insufficient data for XGBoost
#    to learn complex non-linear patterns effectively.
#    Logistic Regression generalizes better on small datasets
#    because it has fewer parameters to fit.
#
# 2. Optuna converged toward simplicity:
#    The best params reflect a deliberately constrained model:
#    - learning_rate : 0.011  (very low → slow, careful learning)
#    - max_depth     : 4      (shallow trees → less complexity)
#    - gamma         : 4.3    (high → suppresses unnecessary splits)
#    Optuna essentially tried to make XGBoost behave like
#    a simpler model to avoid overfitting — which itself
#    suggests the data does not reward complexity.
#
# 3. Linear separability:
#    The relatively strong performance of Logistic Regression
#    suggests that the decision boundary between High and Low ROAS
#    is largely linear in this feature space.
#    XGBoost's strength lies in capturing non-linear interactions,
#    which may not be prominent here.
#
# Conclusion:
#    For this dataset, Logistic Regression is the stronger model.
#    XGBoost tuning confirmed this — not every problem needs
#    a complex model. Model selection should be driven by data,
#    not assumption.
#
# Final Model Comparison:
#    Logistic Regression : ROC-AUC 0.8777 ← winner
#    XGBoost (tuned)     : ROC-AUC 0.8633
#    Difference          : 0.0144
# =============================================================

print("Final Model Comparison")
print("=" * 45)
print(f"{'Model':<25} {'ROC-AUC':>10}")
print("-" * 45)
print(f"{'Logistic Regression':<25} {'0.8777':>10} ← winner")
print(f"{'XGBoost (tuned)':<25} {'0.8633':>10}")
print(f"{'Difference':<25} {'0.0144':>10}")
print()
print("Insight: Small datasets favor simpler models.")
print("         Complexity is not always an advantage.")

# SHAP Feature Importance

In [ ]:
# =============================================================
# SHAP FEATURE IMPORTANCE
#
# SHAP (SHapley Additive exPlanations) explains how much each
# feature contributes to the model's prediction for each sample.
#
# Why SHAP over regular feature importance?
# - Regular importance: "how often was this feature used?"
# - SHAP: "how much did this feature PUSH the prediction
#           toward High or Low ROAS?" → more meaningful
#
# We apply SHAP to both models:
# 1. Logistic Regression (winner)
# 2. XGBoost (tuned)
# And compare which features matter most in each.
#
# Interpretation:
# - High SHAP value → pushes prediction toward High ROAS
# - Low SHAP value  → pushes prediction toward Low ROAS
# - Features near zero → little impact on prediction
# =============================================================

!pip install shap -q
import shap

# ── 1. SHAP for Logistic Regression ──
print("Computing SHAP for Logistic Regression...")

# Use LinearExplainer for Logistic Regression
explainer_lr = shap.LinearExplainer(lr_model, X_train_scaled,
                                     feature_perturbation="interventional")
shap_values_lr = explainer_lr.shap_values(X_test_scaled)

# Summary plot
plt.figure()
shap.summary_plot(
    shap_values_lr,
    X_test,
    feature_names=FEATURES,
    plot_type="bar",
    title="Logistic Regression — Mean |SHAP| Feature Importance",
    show=False
)
plt.title("Logistic Regression — Feature Importance (SHAP)")
plt.tight_layout()
plt.show()

# Beeswarm plot (direction of impact)
plt.figure()
shap.summary_plot(
    shap_values_lr,
    X_test,
    feature_names=FEATURES,
    title="Logistic Regression — SHAP Beeswarm",
    show=False
)
plt.title("Logistic Regression — SHAP Beeswarm (impact direction)")
plt.tight_layout()
plt.show()

# ── 2. SHAP for XGBoost ──
print("Computing SHAP for XGBoost...")

explainer_xgb = shap.TreeExplainer(xgb_tuned)
shap_values_xgb = explainer_xgb.shap_values(X_test)

# Summary plot
plt.figure()
shap.summary_plot(
    shap_values_xgb,
    X_test,
    feature_names=FEATURES,
    plot_type="bar",
    show=False
)
plt.title("XGBoost — Feature Importance (SHAP)")
plt.tight_layout()
plt.show()

# Beeswarm plot
plt.figure()
shap.summary_plot(
    shap_values_xgb,
    X_test,
    feature_names=FEATURES,
    show=False
)
plt.title("XGBoost — SHAP Beeswarm (impact direction)")
plt.tight_layout()
plt.show()

# Anomaly Detection (Z-score)

In [ ]:
# =============================================================
# ANOMALY DETECTION — PHASE 3
#
# Goal: Automatically identify ad campaigns with abnormal ROAS
# performance — either suspiciously high or critically low.
#
# Why this matters:
# - Low ROAS anomalies  → budget is being wasted
# - High ROAS anomalies → scale up or verify data integrity
#
# Method 1: Z-score (statistical approach)
# - Measures how many standard deviations a point is from mean
# - |Z| > 3 → anomaly (covers 99.7% of normal distribution)
# - Fast, interpretable, no training required
#
# Method 2: Isolation Forest (ML approach)
# - Randomly splits features to isolate data points
# - Points that are easy to isolate = anomalies
# - Considers multiple features simultaneously
# - More robust than Z-score for high-dimensional data
# =============================================================

from sklearn.ensemble import IsolationForest
from scipy import stats
import matplotlib.pyplot as plt
import numpy as np

df_anomaly = df.copy()

# ── Method 1: Z-score ──
df_anomaly['roas_zscore'] = stats.zscore(df_anomaly['ROAS'])
df_anomaly['is_anomaly_zscore'] = (df_anomaly['roas_zscore'].abs() > 3).astype(int)

zscore_anomalies = df_anomaly[df_anomaly['is_anomaly_zscore'] == 1]
print(f"=== Z-score Anomalies (|Z| > 3) ===")
print(f"Total anomalies: {len(zscore_anomalies)} / {len(df_anomaly)}")
print(f"Anomaly rate: {len(zscore_anomalies)/len(df_anomaly)*100:.2f}%")
print()
print(zscore_anomalies[['platform', 'campaign_type', 'industry',
                          'country', 'ROAS', 'roas_zscore']].sort_values(
                          'ROAS', ascending=False).head(10))

# ── Method 2: Isolation Forest ──
# Use multiple features for more robust detection
iso_features = ['ROAS', 'CTR', 'CPC', 'ad_spend',
                'conversions', 'CPA', 'revenue']

iso_model = IsolationForest(
    contamination=0.05,  # expect ~5% anomalies
    random_state=42,
    n_estimators=100
)

df_anomaly['is_anomaly_iso'] = iso_model.fit_predict(
    df_anomaly[iso_features])
# IsolationForest: -1 = anomaly, 1 = normal
df_anomaly['is_anomaly_iso'] = (df_anomaly['is_anomaly_iso'] == -1).astype(int)

iso_anomalies = df_anomaly[df_anomaly['is_anomaly_iso'] == 1]
print(f"\n=== Isolation Forest Anomalies ===")
print(f"Total anomalies: {len(iso_anomalies)} / {len(df_anomaly)}")
print(f"Anomaly rate: {len(iso_anomalies)/len(df_anomaly)*100:.2f}%")
print()
print(iso_anomalies[['platform', 'campaign_type', 'industry',
                      'country', 'ROAS', 'ad_spend', 'revenue']
                      ].sort_values('ROAS', ascending=False).head(10))

# ── Agreement between methods ──
df_anomaly['both_anomaly'] = (
    (df_anomaly['is_anomaly_zscore'] == 1) &
    (df_anomaly['is_anomaly_iso'] == 1)
).astype(int)

print(f"\n=== Anomalies flagged by BOTH methods ===")
print(f"Count: {df_anomaly['both_anomaly'].sum()}")
print(df_anomaly[df_anomaly['both_anomaly'] == 1][
    ['platform', 'campaign_type', 'industry',
     'country', 'ROAS', 'ad_spend', 'revenue']
].sort_values('ROAS', ascending=False))

# Anomaly Visualization

In [ ]:
# =============================================================
# ANOMALY DETECTION — VISUALIZATION
#
# We visualize anomalies to extract actionable business insights:
# 1. ROAS distribution with anomaly boundaries
# 2. Anomalies by platform → which platform has most outliers?
# 3. Spend vs Revenue scatter → are high ROAS anomalies
#    driven by low spend or genuinely high revenue?
# 4. Anomaly breakdown by campaign type and industry
#
# Key finding preview:
# - TikTok dominates high-ROAS anomalies
# - Most anomalies are HIGH ROAS (opportunity, not just errors)
# - Both methods agree on 19 high-confidence anomalies
# =============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ── 1. ROAS Distribution with anomaly zones ──
axes[0,0].hist(df_anomaly[df_anomaly['is_anomaly_zscore']==0]['ROAS'],
               bins=50, color='steelblue', alpha=0.7, label='Normal')
axes[0,0].hist(df_anomaly[df_anomaly['is_anomaly_zscore']==1]['ROAS'],
               bins=20, color='#d9534f', alpha=0.9, label='Z-score Anomaly')
axes[0,0].axvline(df_anomaly['ROAS'].mean(), color='black',
                   linestyle='--', label=f'Mean: {df_anomaly["ROAS"].mean():.2f}')
axes[0,0].axvline(df_anomaly['ROAS'].mean() + 3*df_anomaly['ROAS'].std(),
                   color='red', linestyle=':', label='Z=3 boundary')
axes[0,0].set_title('ROAS Distribution — Anomalies Highlighted')
axes[0,0].set_xlabel('ROAS')
axes[0,0].legend()

# ── 2. Anomaly count by platform ──
platform_anomaly = df_anomaly.groupby('platform')['both_anomaly'].sum()
colors = ['#d9534f' if v == platform_anomaly.max() else 'steelblue'
          for v in platform_anomaly]
axes[0,1].bar(platform_anomaly.index, platform_anomaly.values, color=colors)
axes[0,1].set_title('High-Confidence Anomaly Count by Platform\n(flagged by both methods)')
axes[0,1].set_xlabel('Platform')
axes[0,1].set_ylabel('Anomaly Count')
for i, (idx, val) in enumerate(platform_anomaly.items()):
    axes[0,1].text(i, val + 0.2, str(val), ha='center', fontweight='bold')

# ── 3. Ad Spend vs Revenue (anomaly scatter) ──
normal = df_anomaly[df_anomaly['both_anomaly'] == 0]
anomaly = df_anomaly[df_anomaly['both_anomaly'] == 1]

axes[1,0].scatter(normal['ad_spend'], normal['revenue'],
                   alpha=0.3, color='steelblue', s=20, label='Normal')
axes[1,0].scatter(anomaly['ad_spend'], anomaly['revenue'],
                   alpha=0.9, color='#d9534f', s=80,
                   label='Anomaly (both methods)', zorder=5)
axes[1,0].set_title('Ad Spend vs Revenue\n(anomalies in red)')
axes[1,0].set_xlabel('Ad Spend ($)')
axes[1,0].set_ylabel('Revenue ($)')
axes[1,0].legend()

# ── 4. Anomaly rate by platform ──
platform_total   = df_anomaly.groupby('platform')['both_anomaly'].count()
platform_anomaly_rate = (platform_anomaly / platform_total * 100).round(2)

axes[1,1].bar(platform_anomaly_rate.index,
               platform_anomaly_rate.values, color='steelblue')
axes[1,1].set_title('Anomaly Rate by Platform (%)')
axes[1,1].set_xlabel('Platform')
axes[1,1].set_ylabel('Anomaly Rate (%)')
for i, (idx, val) in enumerate(platform_anomaly_rate.items()):
    axes[1,1].text(i, val + 0.1, f'{val}%', ha='center', fontweight='bold')

plt.suptitle('Anomaly Detection — Global Ads Performance',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Business summary ──
print("=== Business Insights ===")
print(f"Total anomalies (both methods): {df_anomaly['both_anomaly'].sum()}")
print(f"Anomaly rate: {df_anomaly['both_anomaly'].mean()*100:.2f}%")
print()
print("Anomalies by platform:")
print(df_anomaly.groupby('platform')['both_anomaly'].sum().sort_values(ascending=False))
print()
high_roas_anomaly = df_anomaly[df_anomaly['both_anomaly']==1]['ROAS']
print(f"Anomaly ROAS range: {high_roas_anomaly.min():.2f} ~ {high_roas_anomaly.max():.2f}")
print(f"Normal  ROAS range: {df_anomaly[df_anomaly['both_anomaly']==0]['ROAS'].min():.2f} ~ {df_anomaly[df_anomaly['both_anomaly']==0]['ROAS'].max():.2f}")

# Anomaly detection without Tiktok


In [ ]:
# =============================================================
# ANOMALY DETECTION — WITHOUT TIKTOK
#
# TikTok dominates high-ROAS anomalies (17 out of 19).
# This raises a question: are TikTok anomalies genuine outliers,
# or is TikTok simply a higher-performing platform overall?
#
# We re-run anomaly detection excluding TikTok to see:
# 1. Does the anomaly pattern change significantly?
# 2. Which platform takes over as the anomaly driver?
# 3. Are Google Ads or Meta Ads hiding anomalies
#    that were overshadowed by TikTok?
# =============================================================

df_no_tiktok = df_anomaly[df_anomaly['platform'] != 'TikTok Ads'].copy()

print(f"Dataset size with TikTok    : {len(df_anomaly)}")
print(f"Dataset size without TikTok : {len(df_no_tiktok)}")
print(f"Removed rows                : {len(df_anomaly) - len(df_no_tiktok)}")
print()

# ── Z-score (without TikTok) ──
df_no_tiktok['roas_zscore'] = stats.zscore(df_no_tiktok['ROAS'])
df_no_tiktok['is_anomaly_zscore'] = (
    df_no_tiktok['roas_zscore'].abs() > 3).astype(int)

print("=== Z-score Anomalies (without TikTok) ===")
print(f"Total: {df_no_tiktok['is_anomaly_zscore'].sum()} / {len(df_no_tiktok)}")
print(f"Rate : {df_no_tiktok['is_anomaly_zscore'].mean()*100:.2f}%")
print()
print(df_no_tiktok[df_no_tiktok['is_anomaly_zscore']==1][
    ['platform', 'campaign_type', 'industry',
     'country', 'ROAS', 'roas_zscore']
].sort_values('ROAS', ascending=False))

# ── Isolation Forest (without TikTok) ──
iso_model_nt = IsolationForest(
    contamination=0.05, random_state=42, n_estimators=100)
df_no_tiktok['is_anomaly_iso'] = iso_model_nt.fit_predict(
    df_no_tiktok[iso_features])
df_no_tiktok['is_anomaly_iso'] = (
    df_no_tiktok['is_anomaly_iso'] == -1).astype(int)

print("=== Isolation Forest Anomalies (without TikTok) ===")
print(f"Total: {df_no_tiktok['is_anomaly_iso'].sum()} / {len(df_no_tiktok)}")
print(f"Rate : {df_no_tiktok['is_anomaly_iso'].mean()*100:.2f}%")

# ── Both methods ──
df_no_tiktok['both_anomaly'] = (
    (df_no_tiktok['is_anomaly_zscore'] == 1) &
    (df_no_tiktok['is_anomaly_iso'] == 1)
).astype(int)

print(f"\n=== Both Methods (without TikTok) ===")
print(f"Count: {df_no_tiktok['both_anomaly'].sum()}")
print()
print(df_no_tiktok[df_no_tiktok['both_anomaly']==1][
    ['platform', 'campaign_type', 'industry',
     'country', 'ROAS', 'ad_spend', 'revenue']
].sort_values('ROAS', ascending=False))

# ── Comparison plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# With TikTok
platform_both = df_anomaly.groupby('platform')['both_anomaly'].sum()
axes[0].bar(platform_both.index, platform_both.values, color='steelblue')
axes[0].set_title('Anomalies by Platform\n(with TikTok)')
axes[0].set_ylabel('Count')
for i, (idx, val) in enumerate(platform_both.items()):
    axes[0].text(i, val + 0.1, str(val), ha='center', fontweight='bold')

# Without TikTok
platform_nt = df_no_tiktok.groupby('platform')['both_anomaly'].sum()
axes[1].bar(platform_nt.index, platform_nt.values, color='#d9534f')
axes[1].set_title('Anomalies by Platform\n(without TikTok)')
axes[1].set_ylabel('Count')
for i, (idx, val) in enumerate(platform_nt.items()):
    axes[1].text(i, val + 0.1, str(val), ha='center', fontweight='bold')

plt.suptitle('Anomaly Comparison — With vs Without TikTok',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Summary comparison ──
print("\n=== Final Comparison ===")
print(f"{'':30} {'With TikTok':>15} {'Without TikTok':>15}")
print("-" * 60)
print(f"{'Total rows':30} {len(df_anomaly):>15} {len(df_no_tiktok):>15}")
print(f"{'Both-method anomalies':30} {df_anomaly['both_anomaly'].sum():>15} {df_no_tiktok['both_anomaly'].sum():>15}")
print(f"{'ROAS max':30} {df_anomaly['ROAS'].max():>15.2f} {df_no_tiktok['ROAS'].max():>15.2f}")
print(f"{'ROAS mean':30} {df_anomaly['ROAS'].mean():>15.2f} {df_no_tiktok['ROAS'].mean():>15.2f}")